![Week 4 Day 3](../assets/banners/w4d3.png)

# Week 4 Day 3 - OCR vs HTR vs Vision LLMs: Turning Answer Sheets into Images

Over the last two days GradeWise sharpened its retrieval: a loop that fixes bad searches, answers that must be grounded, and an evidence retriever that finds what a *correct* answer looks like for every question.

But there's a catch we've been politely ignoring since week 1, when Sam's deadlock answer was *typed* into a Python string. Real exams are **handwritten**. Emma's students write on paper, scan it on the school copier or photograph it with a phone, and hand in a PDF full of pictures. No text layer, no Unicode, just ink.

Today we decide **how** GradeWise will read handwriting, and we prepare the pages for it: rendering PDFs to images, choosing a resolution, and cleaning up a wonky phone photo. Along the way you'll discover something that surprises most engineers: **a vision model is billed by pixels, not pages**.

![Week 4 roadmap](../assets/roadmap/week4.png)

### 🎯 By the end of this lab you will
1. ⚖️ Compare **OCR**, **HTR** and **vision LLMs**, and justify GradeWise's choice
2. 🖼️ Render real (synthetic) handwritten answer sheets to images, right here in the notebook
3. 💰 Measure how **DPI** drives pixels, upload size and **estimated tokens**, and find the sweet spot
4. 🧽 Pre-process a messy phone scan (grayscale, contrast, deskew, size cap) and package it for a multimodal message

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 3 - OCR vs HTR vs Vision LLMs: Choosing a Handwriting Strategy | 🎬 |
| 2 | Day 3 - PDF Pages to Images: DPI, Rotation, Contrast and Pre-Processing | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Today is <b>100% offline</b>: no model is called, so no key is needed. We use the <b>synthetic</b> answer sheets in <code>data/answer-sheets/synthetic/</code>: ten fictional students, drawn in handwriting fonts with realistic scan effects, plus the exact text each one "wrote". If that folder is missing, the next cells say so and draw a small stand-in sheet so you can still follow along.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT
from gradewise.console import success, warn

SYNTHETIC = DATA_DIR / "answer-sheets" / "synthetic"
ALL_SHEETS = {p.stem: p for p in sorted(SYNTHETIC.glob("student_*.pdf"))}  # every sheet on disk
# 📦 The class we loop over and send to a model: all 10 sheets, or student_01 + student_05 (Sam) with DATA_SIZE=small
SHEETS = {sid: ALL_SHEETS[sid] for sid in datasets.synthetic_students() if sid in ALL_SHEETS}
TRUTH_FILE = SYNTHETIC / "transcripts.json"
TRUTH = json.loads(TRUTH_FILE.read_text(encoding="utf-8")) if TRUTH_FILE.exists() else {}

if SHEETS and TRUTH:
    success(
        "Synthetic answer sheets found",
        {
            "Folder": str(SYNTHETIC.relative_to(REPO_ROOT)),
            "Sheets on disk": len(ALL_SHEETS),
            "Sheets in this run": len(SHEETS),
        },
    )
else:
    # ⚠️ Plan B: the sheets aren't there yet (run `uv run scripts/make_synthetic_sheets.py`, or `git pull`).
    #    We draw a small stand-in sheet so the rest of the lab still runs. Its text is our "ground truth".
    from PIL import Image, ImageDraw, ImageFont

    warn("data/answer-sheets/synthetic/ is missing or incomplete: using a small stand-in sheet instead.")
    standin = {
        "Q3": "Dedlock is when a process waits for a long time. Conditions: mutual exclution and circuler wait.",
        "Q4a": "When two threads use the same varible at the same time.",
    }
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/Inkfree.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    pages = []
    for label, qid in [("3)", "Q3"), ("4a)", "Q4a")]:
        img = Image.new("RGB", (1240, 1754), "white")
        draw = ImageDraw.Draw(img)
        words, lines, line = f"{label} {standin[qid]}".split(), [], ""
        for word in words:
            if len(line) + len(word) > 48:
                lines.append(line)
                line = ""
            line += word + " "
        for i, text in enumerate([*lines, line]):
            draw.text((90, 140 + i * 70), text, fill=(28, 45, 140), font=font)
        pages.append(img)
    target = OUTPUT_DIR / "week4" / "student_standin.pdf"
    target.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(target, "PDF", resolution=150, save_all=True, append_images=pages[1:])
    ALL_SHEETS = SHEETS = {"student_standin": target}
    TRUTH = {"student_standin": standin}

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## Part 1: OCR vs HTR vs vision LLMs

Three families of tools can turn a picture of writing into text:

| | **OCR** (e.g. Tesseract) | **HTR** (e.g. TrOCR, Transkribus) | **Vision LLM** (e.g. GPT, Claude, Gemini) |
|---|---|---|---|
| Built for | Printed text | Handwritten *lines* | Whole pages, any content |
| Handwriting | ❌ poor | ✅ good, especially when trained on similar hands | ✅ good to very good |
| Layout (labels, "contd.", margins) | ❌ just characters | ⚠️ needs a separate layout step | ✅ understands the page |
| Structured output (JSON per answer) | ❌ | ❌ | ✅ |
| Diagrams, crossed-out text | ❌ | ❌ | ✅ can describe them |
| Biggest risk | Garbled characters | Needs training data, line segmentation | **"Helpfully" fixing** the student's words, hallucinating |
| Cost | Free, local | Free-ish, needs a GPU | Per token, per page |

![Diagram](../assets/diagrams/33be1d290a61.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart TD
  A[Scanned answer sheet] --> B{Printed or handwritten?}
  B -- printed --> OCR[OCR: fast, free, accurate]
  B -- handwritten --> C{Need layout, labels and JSON?}
  C -- "no, lines of text only" --> HTR[HTR model]
  C -- yes --> VLM[Vision LLM + strict prompt + checks]
```

</details>


GradeWise needs to know **which question** each piece of writing answers, keep crossed-out text separate, mark what it can't read and return JSON. That's a vision LLM's job. The risk column tells us what we must engineer against: tomorrow's prompt is built around **never fixing the student**, and Day 4's confidence checks catch the rest.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">The choice is really a cost/accuracy/effort trade-off. A school with thousands of scripts in one consistent hand-in format might train an HTR model once and run it for free. A small team shipping next month buys accuracy and layout understanding from a vision LLM, and pays per page. Knowing <i>what each page costs</i> (Part 3) is what lets you price the product.</span>
        </td>
    </tr>
</table>

## Part 2: Meet the answer sheets

Each synthetic sheet is a PDF of **scanned images**, one per page. Some students were "scanned" on a clean copier, others photographed with a phone: slightly rotated, unevenly lit, a bit blurry. Here's the class list (📦 just `student_01` and Sam's `student_05` with `DATA_SIZE=small`; the demos below still pick their favourite sheets from everything on disk, since rendering is offline and free):

In [ ]:
from gradewise.transcription.pdf_to_images import page_count

show_table(
    [
        {
            "sheet": name,
            "pages": page_count(path),
            "answers in ground truth": len(TRUTH.get(name, {})),
            "file size": f"{path.stat().st_size / 1024:.0f} KB",
        }
        for name, path in SHEETS.items()
    ]
)

`render_pages()` turns every page into a Pillow image. Under the hood it's PyMuPDF (week 2's PDF library), with `alpha=False` so transparent PDFs don't come out black. Let's look at page 1 of one sheet:

In [ ]:
from gradewise.transcription import render_pages

SHEET = "student_07" if "student_07" in ALL_SHEETS else next(iter(ALL_SHEETS))
pages = render_pages(ALL_SHEETS[SHEET], dpi=100)
print(f"{SHEET}: {len(pages)} pages · page 1 is {pages[0].size[0]}×{pages[0].size[1]} px, mode {pages[0].mode}")

thumb = pages[0].copy()
thumb.thumbnail((560, 800))
display(thumb)

That's what the vision model "sees". Let's zoom into the top of the page, where the first answers are, rendered at 200 DPI:

In [ ]:
from gradewise.transcription import render_page

sharp = render_page(ALL_SHEETS[SHEET], 1, dpi=200)
w, h = sharp.size
display(sharp.crop((0, int(h * 0.04), w, int(h * 0.22))).resize((900, int(900 * 0.18 * h / w))))

## Part 3: DPI, pixels and tokens

When you render a PDF, **DPI** (dots per inch) decides how many pixels you get. An A4 page is 8.27 × 11.69 inches, so the pixel size is simply inches × DPI. More pixels means crisper pen strokes, but also a bigger upload and, often, **more tokens**.

Vision providers count image tokens differently. Two common rules (simplified; check your provider's docs):
- **OpenAI-style tiles** (high detail): shrink to fit 2048 px, then to 768 px on the *short* side, then count 512 px tiles: 85 + 170 per tile. GradeWise's `estimate_image_tokens()` implements this one.
- **Anthropic-style area**: roughly `width × height / 750` tokens, after shrinking anything over ~1568 px on the long edge.

In [ ]:
import io
import time

from gradewise.transcription import preprocess, to_data_url
from gradewise.transcription.pdf_to_images import estimate_image_tokens


def area_tokens(img, long_edge: int = 1568) -> int:
    """A rough 'tokens ≈ pixels / 750' estimate, after the provider's own downscale."""
    w, h = img.size
    scale = min(1.0, long_edge / max(w, h))
    return round(w * scale * h * scale / 750)


def jpeg_kb(img) -> float:
    buffer = io.BytesIO()
    img.convert("L").save(buffer, "JPEG", quality=85)
    return buffer.tell() / 1024

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The next cell renders the same page at 72, 100, 150, 200 and 300 DPI. Going from 100 to 300 DPI gives you <b>9×</b> the pixels. Predict: will the token estimate also grow 9×? What about after GradeWise's size cap (<code>preprocess</code> shrinks the long side to 1600 px)?</span>
        </td>
    </tr>
</table>

In [ ]:
rows = []
for dpi in (72, 100, 150, 200, 300):
    start = time.perf_counter()
    raw = render_page(ALL_SHEETS[SHEET], 1, dpi=dpi)
    capped = preprocess(raw)  # grayscale + auto-contrast + long side ≤ 1600 px
    rows.append(
        {
            "DPI": dpi,
            "raw px": f"{raw.width}×{raw.height}",
            "raw MP": round(raw.width * raw.height / 1e6, 1),
            "after cap": f"{capped.width}×{capped.height}",
            "upload (JPEG)": f"{jpeg_kb(capped):.0f} KB",
            "tiles-rule tokens": estimate_image_tokens(capped),
            "area-rule tokens (raw)": area_tokens(raw),
            "area-rule tokens (capped)": area_tokens(capped),
            "render ms": round((time.perf_counter() - start) * 1000),
        }
    )
show_table(rows)

Three lessons hide in that table:

1. 📈 **Raw pixels explode** with DPI (×9 from 100 to 300), and so do render time and upload size.
2. 🧢 **Providers downscale anyway.** Under the tiles rule, the token count is *flat*: every page ends up as the same handful of tiles. Under the area rule, tokens stop growing once the provider's own cap kicks in. Pixels beyond that are paid for in bandwidth and latency, and the model never sees them.
3. 🎯 So GradeWise renders at **200 DPI** (crisp strokes going *into* the downscale) and then caps the long side at **1600 px** itself. You control the final resolution, instead of leaving it to the provider's resizer.

Don't take the numbers' word for it: look at the strokes. Here is the same patch of writing at three DPIs, blown up to the same size:

In [ ]:
from PIL import Image


def patch(dpi: int, box=(0.07, 0.15, 0.50, 0.21), width: int = 420) -> Image.Image:
    img = render_page(ALL_SHEETS[SHEET], 1, dpi=dpi).convert("L")
    w, h = img.size
    crop = img.crop((int(box[0] * w), int(box[1] * h), int(box[2] * w), int(box[3] * h)))
    return crop.resize((width, int(width * crop.height / crop.width)), Image.Resampling.NEAREST)


strip = [patch(dpi) for dpi in (72, 100, 200)]
canvas = Image.new("L", (sum(p.width for p in strip) + 20 * (len(strip) - 1), strip[0].height), 255)
x = 0
for p in strip:
    canvas.paste(p, (x, 0))
    x += p.width + 20
print("        72 DPI                                   100 DPI                                  200 DPI")
display(canvas)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Resolution matters most for <b>small things</b>: a subscript in <code>2^13</code>, a decimal point in an IP address, the difference between <code>1</code> and <code>l</code>. Those are exactly where a mark can hinge on one character. Below ~100 DPI thin strokes break up; above ~200 DPI you mostly pay for pixels the model will never see. And the pricing rules here are simplified and change over time: <b>always check your provider's vision docs</b>.</span>
        </td>
    </tr>
</table>

## Part 4: Pre-processing a messy scan

`student_10` photographed their sheet on a desk: tilted, dim in one corner, a bit blurry, with a dark background around the paper. `preprocess()` does four cheap things before a page goes to the model:

1. **Grayscale**: colour rarely helps with pen on paper, and it shrinks the JPEG
2. **Auto-contrast**: faint ink and grey scanner backgrounds become dark ink on white (`cutoff=1` ignores the darkest/lightest 1%, so one speck of dust can't skew it)
3. **Deskew** (optional): straighten a tilted page
4. **Size cap**: long side ≤ `max_side` (1600 px by default)

In [ ]:
MESSY = "student_10" if "student_10" in ALL_SHEETS else SHEET
before = render_page(ALL_SHEETS[MESSY], 1, dpi=150)
after = preprocess(before)


def side_by_side(*images, height: int = 620) -> Image.Image:
    scaled = [im.convert("RGB").resize((int(im.width * height / im.height), height)) for im in images]
    out = Image.new("RGB", (sum(s.width for s in scaled) + 24 * (len(scaled) - 1), height), "white")
    x = 0
    for s in scaled:
        out.paste(s, (x, 0))
        x += s.width + 24
    return out


print(
    f"before: {before.size} {before.mode} · {jpeg_kb(before):.0f} KB   →   after: {after.size} {after.mode} · "
    f"{jpeg_kb(after):.0f} KB"
)
display(side_by_side(before, after))

The ink on the right is darker and the yellow light is gone: much easier to read. But look at the file sizes: on this photo the JPEG got **bigger**, because auto-contrast also stretched the camera's grainy noise, and noise is expensive to compress. On a clean copier scan it shrinks. The lesson is the one you'll hear all week: **measure, don't assume**.

### Deskewing

`estimate_skew()` uses a classic trick called a **projection profile**: lines of handwriting are horizontal bands of ink, so if you add up the dark pixels in every row, a straight page gives sharp peaks and valleys while a tilted page smears them out. It tries small angles and keeps the one with the spikiest profile.

To test it fairly, we take a neat page and tilt it by a known amount:

In [ ]:
from gradewise.transcription.pdf_to_images import estimate_skew

NEAT = "student_01" if "student_01" in ALL_SHEETS else SHEET
straight = preprocess(render_page(ALL_SHEETS[NEAT], 1, dpi=100))
tilted = straight.rotate(-3, fillcolor=255, expand=True)  # 3° clockwise, like a careless scan

print(f"estimated skew of the straight page: {estimate_skew(straight):+.1f}°")
print(f"estimated skew of the tilted page:   {estimate_skew(tilted):+.1f}°  (rotate by this to fix it)")
fixed = preprocess(tilted, deskew=True)
display(side_by_side(tilted, fixed, height=500))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Deskew is <b>off by default</b> in <code>preprocess()</code>. Why might that be a good default? Think about CPU time per page for a class of 200 scripts, how well vision models already cope with a 2° tilt, and what could go wrong if the estimate is wrong on a page that is mostly a diagram.</span>
        </td>
    </tr>
</table>

## Part 5: Packaging a page for the model

A chat model takes images inside a message, as **content blocks**. `vision.py` encodes the page as base64 JPEG and puts it next to the instructions as a LangChain 1.x standard block, `{"type": "image", "base64": ..., "mime_type": "image/jpeg"}` (`image_block()`). We send base64 rather than a `data:` URL because the Google package tries to download any URL. `to_data_url()` still shows what the bytes look like. Every provider package translates the standard block for you, which every provider package translates for you (OpenAI's `image_url`, Anthropic's `source`, ...):

In [ ]:
from gradewise.transcription.vision import build_messages

url = to_data_url(after)
print(f"data URL: {url[:40]}… ({len(url) / 1024:.0f} KB of base64 text)")

messages = build_messages(after, page_number=1, question_ids=["Q1", "Q2a", "Q2b", "Q3"])
for message in messages:
    if isinstance(message.content, str):
        print(f"\n{type(message).__name__}: {message.content[:160]}…")
    else:
        print(f"\n{type(message).__name__}:")
        for block in message.content:
            shown = {k: (v[:50] + "…" if isinstance(v, str) and len(v) > 60 else v) for k, v in block.items()}
            print("   ", shown)

💡 **Why JPEG?** At quality 85, a scanned page is typically 5–10× smaller than PNG and looks the same to the model. If you ever see compression smudges around very thin strokes, `to_data_url(img, fmt="PNG")` is one argument away.

That `SystemMessage` is tomorrow's star: the **verbatim transcription prompt**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Everything today lives in <code>gradewise/transcription/pdf_to_images.py</code>: <code>render_pages</code>, <code>render_page</code>, <code>preprocess</code>, <code>estimate_skew</code>, <code>to_data_url</code> and <code>estimate_image_tokens</code>. Open it and read the "DPI vs TOKEN COST" comment at the top: <code>DEFAULT_DPI = 200</code> and <code>MAX_SIDE = 1600</code> are the two knobs you just tuned by hand. On Day 5 they become the first node of the transcription subgraph.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- ⚖️ compared **OCR, HTR and vision LLMs**, and saw why GradeWise uses a vision LLM, with guard rails
- 🖼️ rendered real handwritten answer sheets to images and zoomed into the strokes
- 💰 measured how DPI drives pixels, upload size and tokens, and why **200 DPI + a 1600 px cap** is the sweet spot
- 🧽 cleaned a messy phone photo (grayscale, auto-contrast, deskew) and packed it into a multimodal message

**Tomorrow (Day 4):** we send those images to a vision model with a prompt whose first rule is *never fix the student*. Then we map every answer to its question, even when it's labelled "Ans 4(ii)" or continues on the next page, and we put a number on how much to trust each transcription.

⏭ **Next:** `week4/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Price a whole class. For every synthetic sheet, render each page at 150, 200 and 300 DPI, apply <code>preprocess()</code> and add up both token estimates (tiles rule and area rule). Assume the default model's input price (<code>gpt-6-luna</code>: $0.10 per million tokens) and about 900 prompt tokens per page. What does it cost to read the class (all 10 sheets, or 2 with <code>DATA_SIZE=small</code>) at each DPI? And what changes if you raise the size cap to 2400 px? Solution: <code>week4/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• OpenAI vision guide (image inputs and token costs): <a href="https://platform.openai.com/docs/guides/images-vision">platform.openai.com/docs/guides/images-vision</a><br/>• Anthropic vision docs: <a href="https://docs.anthropic.com/en/docs/build-with-claude/vision">docs.anthropic.com/en/docs/build-with-claude/vision</a><br/>• LangChain multimodal messages: <a href="https://docs.langchain.com/oss/python/langchain/messages">docs.langchain.com/oss/python/langchain/messages</a><br/>• TrOCR, a transformer HTR model: <a href="https://arxiv.org/abs/2109.10282">arxiv.org/abs/2109.10282</a></span>
        </td>
    </tr>
</table>